## Решение заданий с помощью Spark DataFrame API

Установим Spark подключение и проверим его.

In [1]:
# import pyspark as spark

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, max as spark_max, sum as spark_sum, when, row_number, desc, expr
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Hadoop_Assignment_VSCode") \
    .enableHiveSupport() \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/10 20:29:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
spark.sql("USE s_yudin")

26/06/10 20:29:34 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


DataFrame[]

In [3]:
#test
spark.sql("""
    SELECT * FROM dim_cat LIMIT 5
""").show()

+------+--------------------+
|cat_id|       category_name|
+------+--------------------+
|     1|                  $1|
|     2|                 $10|
|     3|          $10, Eagle|
|     4|                $100|
|     5|$2.50, Quarter Eagle|
+------+--------------------+



In [4]:
# Загружаем данные в DataFrame
df = spark.table("ebay_listings_optimized")

### 1

Рассчитать количество товаров, среднюю цену и максимальную цену для каждой category_name

---

In [6]:
result1 = df.filter("price > 0") \
    .groupBy("category_name") \
    .agg(
        count("*").alias("item_count"),
        avg("price").alias("avg_price"),
        spark_max("price").alias("max_price")
    ) \
    .orderBy("category_name")

result1.show()

+--------------------+----------+------------------+---------+
|       category_name|item_count|         avg_price|max_price|
+--------------------+----------+------------------+---------+
|                NULL|       185|50.944810810810786|   238.06|
|                  $1|        62|147.39661290322584|   990.76|
|                 $10|         8|          280.8275|   977.98|
|          $10, Eagle|        23| 31.97826086956522|    34.99|
|                $100|         7|115.11142857142859|   604.72|
|$2.50, Quarter Eagle|         2|             57.97|    58.28|
|   1 Pfennig ab 1950|         1|             23.12|    23.12|
|10-Euro-Gedenkmünzen|         1|             21.79|    21.79|
|                1000|       101| 61.96999999999999|    158.8|
|              100005|         4|           30.0925|    54.38|
|              100011|         1|             71.99|    71.99|
|              100012|         2|             32.87|    37.93|
|              100058|        38| 90.42894736842106|   

### 2

Выявить связь между статусом продавца и условиями доставки. Вывести топ-10 стран (location_country), в которых средний процент положительных отзывов (seller_feedback_percentage) у «топ-продавцов» (top_rated_buying_experience = true) выше всего

---

In [7]:
result2 = df.filter((col("top_rated_buying_experience") == True) & col("seller_feedback_percentage").isNotNull()) \
    .groupBy("location_country") \
    .agg(avg("seller_feedback_percentage").alias("avg_positive_feedback")) \
    .orderBy(col("avg_positive_feedback").desc()) \
    .limit(10)

result2.show()

+----------------+---------------------+
|location_country|avg_positive_feedback|
+----------------+---------------------+
|              ME|                100.0|
|              MO|                100.0|
|              MK|                100.0|
|              MW|                100.0|
|              NA|                100.0|
|              MU|                100.0|
|              SN|                100.0|
|              KG|                100.0|
|              UY|                100.0|
|              CI|                100.0|
+----------------+---------------------+



### 3

Создать новый столбец total_price_usd, который суммирует price и shipping_cost. Вывести список товаров, где эта сумма превышает 500 USD, а доставка является фиксированной (shipping_cost_type = 'FIXED')

---

In [8]:
result3 = df.withColumn("total_price_usd", col("price") + col("shipping_cost")) \
    .filter((col("total_price_usd") > 500) & (col("shipping_cost_type") == "FIXED")) \
    .select("itemid", "title", "price", "shipping_cost", "total_price_usd", "shipping_cost_type") \
    .orderBy(col("total_price_usd").desc())

result3.show()

+------------+---------------------+---------+-------------+-----------------+------------------+
|      itemid|                title|    price|shipping_cost|  total_price_usd|shipping_cost_type|
+------------+---------------------+---------+-------------+-----------------+------------------+
|227332977407| Lego Pokemon 7215...|999999.99|         75.0|       1000074.99|             FIXED|
|168308219349| bricks porsche rs...|1000000.0|         15.0|        1000015.0|             FIXED|
|236675213386| Orange & Red Sush...|999999.99|         10.0|       1000009.99|             FIXED|
|157780805826| Pokemon Base Set ...|1000000.0|         6.99|       1000006.99|             FIXED|
|306894108935| 2002 POKEMON LEGE...|1000000.0|         5.99|       1000005.99|             FIXED|
|406831816952| Ty 4005 Beanie Ba...|1000000.0|          0.0|        1000000.0|             FIXED|
|406825938053| Ty 4005 Beanie Ba...|1000000.0|          0.0|        1000000.0|             FIXED|
|406831773685| Ty Be

# 4

В каждой подкатегории (sub_category_name) найти по одному продавцу с самым высоким показателем seller_feedback_score

---

In [9]:
window_spec = Window.partitionBy("sub_category_name").orderBy(col("seller_feedback_score").desc(), col("seller_name"))
result4 = df.filter(col("seller_feedback_score").isNotNull()) \
    .withColumn("rn", row_number().over(window_spec)) \
    .filter(col("rn") == 1) \
    .select("sub_category_name", "seller_name", "seller_feedback_score") \
    .orderBy("sub_category_name")

result4.show()

+-----------------+------------------+---------------------+
|sub_category_name|       seller_name|seller_feedback_score|
+-----------------+------------------+---------------------+
|             NULL|          the_nile|              1274042|
|$20, Double Eagle|       fb64-4glrx-|                    0|
|                1|       jLuLeCbjRSa|              5680850|
|            10033|       CV4X1GnWR6O|               696685|
|            10063|   zs_zweiradteile|               880616|
|           101925|        wuxianshen|                    9|
|            10285|         mohfaisal|                    6|
|           103430|        city-green|               689963|
|           103431|       yj3mEOG2RoO|                  542|
|             1039|arcadia_modellismo|                17181|
|           105559|       zoraoptwtpc|                32148|
|             1059|       iWiCELLQSL2|              1205604|
|           106256|           xyplus6|                 3556|
|           106981|     

# 5

Посчитать количество пропущенных значений (NULL) в колонке estimated_delivery_days для каждой даты выгрузки (snapshot_dt). Вывести только те даты, где количество пропусков больше 10% от общего числа записей за этот день.

---

In [10]:
result5 = df.filter(col("snapshot_dt").isNotNull()) \
    .groupBy("snapshot_dt") \
    .agg(
        count("*").alias("total_rows"),
        spark_sum(when(col("estimated_delivery_days").isNull(), 1).otherwise(0)).alias("null_count")
    ) \
    .withColumn("null_percentage", (col("null_count") / col("total_rows")) * 100) \
    .filter(col("null_percentage") > 10) \
    .orderBy("snapshot_dt")

result5.show()

+-----------+----------+----------+------------------+
|snapshot_dt|total_rows|null_count|   null_percentage|
+-----------+----------+----------+------------------+
| 2026-02-28|     26223|      7689|  29.3215879190024|
| 2026-03-01|     61907|     22671|36.621060623192854|
| 2026-03-02|       127|        29| 22.83464566929134|
| 2026-03-03|    102470|     22743| 22.19478871864936|
| 2026-03-04|    145406|     36247| 24.92813226414316|
| 2026-03-05|    151337|     39292| 25.96324758651222|
| 2026-03-06|    151683|     34208|22.552296565864335|
| 2026-03-07|    138809|     34068| 24.54307717799278|
| 2026-03-08|    137803|     37026| 26.86879095520417|
| 2026-03-09|    112467|     27159|24.148416868948225|
| 2026-03-10|    118220|     27307|23.098460497377772|
| 2026-03-15|    137824|     39891|28.943435105641978|
| 2026-03-16|    132314|     33485|25.307223725380535|
| 2026-03-17|    150102|     36265| 24.16023770502725|
| 2026-03-18|    140551|     35543|25.288329503169667|
| 2026-03-